# CcMart — Machine Learning Pipeline
**ITCS 6190/8190 Cloud Computing for Data Analysis**

Three MLlib models:
1. **Random Forest** — Payment success classification
2. **KMeans** — Customer segmentation (k=5)
3. **ALS** — Collaborative filtering product recommender

## Setup

In [ ]:
from pyspark.sql import SparkSession
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StringIndexer, StandardScaler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.clustering import KMeans
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import (
    MulticlassClassificationEvaluator, BinaryClassificationEvaluator,
    ClusteringEvaluator, RegressionEvaluator
)

spark = SparkSession.builder.appName('CcMart-ML').getOrCreate()
print('MLlib ready.')

## Model 1 — Random Forest: Payment Status Prediction
**Why Random Forest?** Handles non-linearity, mixed data types, prevents overfitting via bagging — better than Logistic Regression for this dataset.

In [ ]:
txns = spark.read.parquet('../data/processed/transactions')
customers = spark.read.parquet('../data/processed/customers')

# Join for demographic features
data = txns.join(customers, 'customer_id', 'left')

# Build pipeline
payment_idx   = StringIndexer(inputCol='payment_method', outputCol='pay_idx')
device_idx    = StringIndexer(inputCol='device_type', outputCol='dev_idx', handleInvalid='keep')
label_indexer = StringIndexer(inputCol='payment_status', outputCol='label')
assembler = VectorAssembler(
    inputCols=['transaction_total', 'pay_idx', 'dev_idx', 'num_items'],
    outputCol='features', handleInvalid='skip'
)
rf = RandomForestClassifier(numTrees=100, maxDepth=5, seed=42)

pipeline = Pipeline(stages=[payment_idx, device_idx, label_indexer, assembler, rf])
train, test = data.randomSplit([0.8, 0.2], seed=42)
model = pipeline.fit(train)
preds = model.transform(test)

acc = MulticlassClassificationEvaluator(metricName='accuracy').evaluate(preds)
auc = BinaryClassificationEvaluator(metricName='areaUnderROC').evaluate(preds)
print('Accuracy:', round(acc, 4))
print('AUC-ROC: ', round(auc, 4))

## Model 2 — KMeans Customer Segmentation (k=5)
**Why KMeans?** Unsupervised — discovers hidden clusters without labels. k=5 chosen by Elbow method + Silhouette score.

In [ ]:
from pyspark.sql.functions import sum as _sum, count, datediff, current_date, max as _max

# Build RFM features
rfm = txns.groupBy('customer_id').agg(
    _sum('transaction_total').alias('monetary'),
    count('transaction_id').alias('frequency'),
    datediff(current_date(), _max('transaction_date')).alias('recency')
).fillna(0)

assembler2 = VectorAssembler(inputCols=['monetary','frequency','recency'], outputCol='features')
scaler = StandardScaler(inputCol='features', outputCol='scaled_features')
kmeans = KMeans(featuresCol='scaled_features', k=5, seed=42)
pipeline2 = Pipeline(stages=[assembler2, scaler, kmeans])
model2 = pipeline2.fit(rfm)
preds2 = model2.transform(rfm)

silhouette = ClusteringEvaluator(featuresCol='scaled_features').evaluate(preds2)
print('Silhouette Score:', round(silhouette, 4))
preds2.groupBy('prediction').count().orderBy('prediction').show()

## Model 3 — ALS Collaborative Filtering Recommender
**Why ALS?** Learns from implicit signals (views, clicks, purchases). Discovers latent user-item preferences. Content-based filtering cannot learn cross-user patterns.

In [ ]:
# Build implicit interaction matrix
from pyspark.sql.functions import when, lit
from pyspark.ml.feature import StringIndexer as SI

click_data = spark.read.parquet('../data/processed/click_stream')

# Score interactions implicitly
interactions = click_data.withColumn('score',
    when(col('event_type') == 'booking_page', 5)
    .when(col('event_type') == 'add_to_cart', 3)
    .when(col('event_type') == 'wishlist', 2)
    .otherwise(1)
)

cust_idx = SI(inputCol='customer_id', outputCol='cust_idx')
prod_idx = SI(inputCol='product_id', outputCol='prod_idx')
interactions_indexed = Pipeline(stages=[cust_idx, prod_idx]).fit(interactions).transform(interactions)

als = ALS(userCol='cust_idx', itemCol='prod_idx', ratingCol='score',
          implicitPrefs=True, nonnegative=True, coldStartStrategy='drop', seed=42)
train3, test3 = interactions_indexed.randomSplit([0.8, 0.2], seed=42)
als_model = als.fit(train3)

rmse = RegressionEvaluator(metricName='rmse', labelCol='score', predictionCol='prediction').evaluate(als_model.transform(test3))
print('RMSE:', round(rmse, 4))

# Top-5 recommendations for all users
recs = als_model.recommendForAllUsers(5)
recs.show(5, truncate=False)

## Run the full ML pipeline
```bash
python ../src/ml_pipeline.py
```
Outputs saved to `outputs/ml/`